# Volume 13, Chapter 10: Multiplicative functions (Python companion)

Exercises first, solutions at the end. We sieve the classical multiplicative functions up to $10^7$, test Delange's Euler products and Wirsing's theorem, draw the landscape of pretentious distances $t\mapsto\mathbb D(\mu,n^{it};x)^2$, and compare Halász's bound with the truth. The computer science application: **Möbius disjointness for automatic sequences**, sequences produced by finite automata reading the binary digits of $n$ (Thue–Morse, Rudin–Shapiro), which Sarnak's conjecture, proved for them by Müllner, says cannot correlate with $\mu$. Requires `numpy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath as mp

N = 10**7
t0 = time.time()
isp = np.ones(N + 1, dtype=bool); isp[:2] = False
for i in range(2, int(N**0.5) + 1):
    if isp[i]: isp[i*i::i] = False
primes = np.nonzero(isp)[0]
mu = np.ones(N + 1, dtype=np.int8); mu[0] = 0
lam = np.ones(N + 1, dtype=np.int8); lam[0] = 0
for p in primes:
    mu[p::p] *= -1
    if p * p <= N: mu[p*p::p*p] = 0
    pk = p
    while pk <= N:
        lam[pk::pk] *= -1; pk *= p
print(f"sieved mu and lambda up to {N:.0e} in {time.time()-t0:.1f} s")

## Exercises

**P10.1 (★) Mean values.** Compute $\frac1x\sum_{n\le x}f(n)$ at $x=10^7$ for $f=\mu,\lambda,\mu^2$, for $f(n)=\frac{\varphi(n)}n$ and for the multiplicative $g$ with $g(p^k)=1-\frac2p$. Compare the last three with the Euler products of Theorem 10.8 (compute them first by hand: they are $\prod_p(1-\frac1{p^2})$, $\prod_p(1-\frac1{p^2})$ and $\prod_p(1-\frac2{p^2})$).

**P10.2 (★) The spiral.** For $f(n)=n^{i}$, compare $\frac1x\sum_{n\le x}n^{i}$ with $\frac{x^{i}}{1+i}$ for $x=10^3,\dots,10^6$ (Exercise 10.1 and Figure 10.1). How fast does the difference shrink?

**P10.3 (★★) The distance landscape.** Plot $t\mapsto\mathbb D(\mu,n^{it};x)^2$ for $x=10^6$ and $|t|\le40$, find its minimum, and compare with the prediction of Lemma 10.3(b): $\mathbb D(\mu,n^{it};x)^2=2\sum_{p\le x}\frac1p-\mathbb D(1,n^{it};x)^2\approx\log\log x+\log|\zeta(1+\frac1{\log x}+it)|+c$. Where are the dips, and why?

**P10.4 (★★) Halász against the truth.** Corollary 10.11 says $\frac1x|\sum_{n\le x}\mu(n)|\ll\frac{(\log\log x)^{5/4}}{(\log x)^{1/4}}$. Tabulate both sides for $x=10^3,\dots,10^7$. Then look at the slow Wirsing case: the indicator $b$ of sums of two squares has mean $0$, and Landau proved $\sum_{n\le x}b(n)\sim K\frac x{\sqrt{\log x}}$ with $K=0.764223\ldots$. How close is $\frac{\sqrt{\log x}}x\sum_{n\le x}b(n)$ to $K$ at $10^7$?

**P10.5 (★★, computer science) Automata cannot fake Möbius.** A sequence is *$2$-automatic* if $a(n)$ is computed by a finite automaton reading the binary digits of $n$. Build automata for Thue–Morse $t(n)=(-1)^{s_2(n)}$ ($s_2$ = sum of binary digits) and Rudin–Shapiro $r(n)=(-1)^{\#\{\text{occurrences of }11\text{ in }n\}}$, and compute the correlations $\frac1x\sum_{n\le x}\mu(n)a(n)$ for these, for a periodic sequence, for $e(n\sqrt2)$, and, for contrast, for $a=\mu$ and $a=\lambda$. Which sequences correlate with $\mu$, and what distinguishes them?

## Solutions

### P10.1

In [ ]:
phi_over_n = np.ones(N + 1); g = np.ones(N + 1)
for p in primes:
    phi_over_n[p::p] *= 1 - 1 / p
    g[p::p] *= 1 - 2 / p
def euler(fac, P=10**6):
    ps = primes[primes <= P].astype(float)
    return float(np.exp(np.sum(np.log(fac(ps)))))
rows = [("mu", mu[1:].sum() / N, 0.0), ("lambda", lam[1:].sum() / N, 0.0),
        ("mu^2", (mu[1:] != 0).sum() / N, 6 / math.pi**2),
        ("phi(n)/n", phi_over_n[1:].sum() / N, euler(lambda p: 1 - 1 / p**2)),
        ("g", g[1:].sum() / N, euler(lambda p: 1 - 2 / p**2))]
for name, avg, pred in rows:
    print(f"{name:9s} average {avg: .7f}   Euler product {pred:.7f}")

The Euler factors first. For $f=\mu^2$: $(1-\frac1p)(1+\frac1p)=1-\frac1{p^2}$. For $f(n)=\frac{\varphi(n)}n$, $f(p^k)=1-\frac1p$ for $k\ge1$, so the factor is $(1-\frac1p)\bigl(1+(1-\frac1p)\frac{1/p}{1-1/p}\bigr)=(1-\frac1p)(1+\frac1p)$, the same. For $g$: $(1-\frac1p)\bigl(1+\frac{1-2/p}{p-1}\bigr)=\frac{p-1}p+\frac{p-2}{p^2}=1-\frac2{p^2}$. All three satisfy $\sum_p\frac{|1-f(p)|}p<\infty$, so Theorem 10.8 applies.

The output agrees to seven digits for $\frac{\varphi(n)}n$ and $g$, and to six for $\mu^2$ ($0.6079291$ against $\frac6{\pi^2}=0.6079271$). The difference is instructive: $\frac{\varphi(n)}n=\sum_{d\mid n}\frac{\mu(d)}d$ has an error term $O(\frac{\log x}x)$, while for $\mu^2$ the error is of order $x^{-1/2}$, here $2\cdot10^{-6}$. The constant $\prod_p(1-\frac2{p^2})=0.3226\ldots$ is also the density of the $n$ for which $n$ and $n+1$ are both squarefree. For $\mu$ and $\lambda$ Wirsing's theorem says the mean is $0$, and indeed $\frac1x\sum\mu=1.0\cdot10^{-4}$ and $\frac1x\sum\lambda=-0.8\cdot10^{-4}$ at $10^7$.

### P10.2

In [ ]:
M = 10**6
n = np.arange(1, M + 1)
s = np.cumsum(np.exp(1j * np.log(n)))
for x in [10**3, 10**4, 10**5, 10**6]:
    avg = s[x - 1] / x
    pred = np.exp(1j * math.log(x)) / (1 + 1j)
    print(f"x=1e{round(math.log10(x))}: average {avg:.6f}, x^i/(1+i) = {pred:.6f}, difference {abs(avg - pred):.2e}")

The difference shrinks roughly by a factor of $10$ per decade of $x$: $8\cdot10^{-4}$, $7\cdot10^{-5}$, $2.5\cdot10^{-6}$, $9\cdot10^{-7}$. This matches Exercise 10.1: $\sum_{n\le x}n^{it}=\frac{x^{1+it}}{1+it}+O(1+|t|)$, so the averages differ from $\frac{x^{it}}{1+it}$ by $O(\frac1x)$ (the irregular steps come from the bounded oscillating term). The averages never settle: they run around the circle of radius $\frac1{|1+i|}=0.7071$, once each time $\log x$ grows by $2\pi$, that is each time $x$ is multiplied by $e^{2\pi}\approx535$.

### P10.3

In [ ]:
x = 10**6
ps = primes[primes <= x].astype(float); lp = np.log(ps); w = 1 / ps
ts = np.arange(-40, 40.0001, 0.01)
D2 = np.empty(len(ts))
for i in range(0, len(ts), 500):
    D2[i:i+500] = (np.cos(np.outer(ts[i:i+500], lp)) * w).sum(axis=1) + w.sum()
i0 = np.argmin(D2)
print(f"sum 1/p up to 1e6 = {w.sum():.4f};  D(mu,1)^2 = {D2[len(ts)//2]:.4f}")
print(f"minimum of D(mu, n^it)^2 over |t|<=40: {D2[i0]:.4f} at t = {ts[i0]:.2f}")
# local minima, and the prediction loglog x + log|zeta(1+sigma+it)| + c
sig = 1 / math.log(x)
pred = lambda t: math.log(math.log(x)) + math.log(abs(mp.zeta(1 + sig + 1j * t)))
c = D2[np.searchsorted(ts, 20.0)] - pred(20.0)
mins = [j for j in range(1, len(ts) - 1) if D2[j] < D2[j-1] and D2[j] < D2[j+1] and ts[j] > 0]
best = sorted(mins, key=lambda j: D2[j])[:5]
for j in sorted(best):
    print(f"  dip at t = {ts[j]:6.2f}: D^2 = {D2[j]:.4f}, prediction {pred(ts[j]) + c:.4f}")
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3)); plt.plot(ts, D2, lw=0.7)
plt.xlabel("t"); plt.ylabel("D(mu, n^it; 1e6)^2"); plt.show()

At $t=0$ the distance is maximal: $\mathbb D(\mu,1;10^6)^2=2\sum_{p\le10^6}\frac1p=5.77$. The landscape then drops, and the deepest dips over $0<t\le40$ sit at $t=14.09$, $21.06$, $24.99$, $30.57$ and $32.83$, the first being the minimum, $1.64$. These are close to the ordinates $14.13$, $21.02$, $25.01$, $30.42$, $32.94$ of the first zeros of $\zeta$. The reason is Lemma 10.3(b): $\mathbb D(\mu,n^{it};x)^2$ is $\log\log x+\log|\zeta(1+\frac1{\log x}+it)|$ up to a bounded error, and $|\zeta|$ on the line $\Re s=1$ is small just to the right of a zero. The printed predictions, with the constant fitted at $t=20$, find the right dips but are off by $0.07$ to $0.3$: the bounded error of Lemma 10.3(b) is bounded, not constant.

So the Möbius function "pretends" most to be $n^{14.1i}$, but not much: $1.64$ is still a sizeable distance, $e^{-1.64}\approx0.19$, and as $x$ grows the whole landscape rises like $\log\log x$ for bounded $t$ (Corollary 10.11). Halász's theorem has to look at every such dip up to $|t|\le2(\log x)^3$; a dip that stayed bounded would be a zero of $\zeta$ on the line $\Re s=1$, which is the prime number theorem in disguise.

### P10.4

In [ ]:
cm = np.cumsum(mu.astype(np.int64))
for k in range(3, 8):
    x = 10**k; L = math.log(x)
    print(f"x=1e{k}: |M(x)|/x = {abs(cm[x])/x:.2e}   Halasz shape (loglog x)^(5/4)/(log x)^(1/4) = {math.log(L)**1.25 / L**0.25:.3f}")
b = np.zeros(N + 1, dtype=bool)
for a in range(int(N**0.5) + 1):
    bs = np.arange(a, int((N - a*a)**0.5) + 1)
    b[a*a + bs*bs] = True
b[0] = False
cb = np.cumsum(b)
for k in range(3, 8):
    x = 10**k
    print(f"x=1e{k}: B(x)/x = {cb[x]/x:.4f}   B(x) sqrt(log x)/x = {cb[x]*math.sqrt(math.log(x))/x:.4f}")

The truth is tiny ($\frac1x|M(x)|=1.0\cdot10^{-4}$ at $10^7$) and Halász's bound is larger than $1$ throughout, so as a numerical estimate it says nothing here. The shape $\frac{(\log\log x)^{5/4}}{(\log x)^{1/4}}$ even increases at first; it drops below $1$ only when $x$ has about $10^5$ digits, and tends to $0$ extremely slowly. This is the price of a method that uses nothing but $|f|\le1$ and multiplicativity: it treats $\mu$ like every other function far from all $n^{it}$. The zeros of $\zeta$ (Volume 5) give $\frac1xM(x)\ll e^{-c\sqrt{\log x}}$, and the Riemann hypothesis would give $x^{-1/2+\varepsilon}$, which is the size seen in the table.

For the sums of two squares, $\frac{B(x)}x$ goes to $0$ as Wirsing's theorem predicts, but slowly: $0.33$, $0.27$, $0.24$, $0.22$, $0.20$. The normalized count $\frac{B(x)\sqrt{\log x}}x$ decreases towards Landau's constant $K=0.7642$, reaching $0.7971$ at $10^7$. The convergence is slow because the next term of the asymptotic expansion is smaller only by a factor of order $\frac1{\log x}$.

### P10.5

In [ ]:
# A 2-automaton: states, a transition table delta[state][bit], an output map; it reads n's binary digits, most significant first.
def run(delta, out, n):
    s = 0
    for bit in bin(n)[2:]:
        s = delta[s][int(bit)]
    return out[s]
TM = ([[0, 1], [1, 0]], [1, -1])                      # state = parity of the number of 1s
RS = ([[0, 1], [0, 2], [3, 1], [3, 2]], [1, 1, -1, -1])  # state = (parity of 11-blocks, last bit)
X = 10**6
nn = range(1, X + 1)
tm = np.array([run(*TM, k) for k in nn]); rs = np.array([run(*RS, k) for k in nn])
# check the automata against the definitions
assert all(tm[k-1] == (-1)**bin(k).count("1") for k in range(1, 5000))
assert all(rs[k-1] == (-1)**sum(1 for j in range(len(bin(k)) - 3) if bin(k)[2+j:4+j] == "11") for k in range(1, 5000))
m = mu[1:X+1].astype(float); n = np.arange(1, X + 1)
seqs = {"Thue-Morse": tm, "Rudin-Shapiro": rs, "(-1)^n": (-1.0)**n,
        "period 3: 1,1,-2": np.array([1, 1, -2])[n % 3],
        "Re e(n sqrt 2)": np.cos(2 * math.pi * n * math.sqrt(2)),
        "mu itself": m, "lambda": lam[1:X+1].astype(float)}
for name, a in seqs.items():
    print(f"{name:18s} (1/x) sum mu(n) a(n) at x=1e6: {np.dot(m, a)/X: .5f}")

The automata check out against the definitions. Thue–Morse needs two states, the parity of the number of $1$s read; Rudin–Shapiro needs four, the parity of the number of blocks $11$ read so far together with the last digit read. All five "structured" sequences, Thue–Morse, Rudin–Shapiro, $(-1)^n$, the period-$3$ sequence and $\cos(2\pi n\sqrt2)$, give correlations below $10^{-3}$ at $x=10^6$, the same size as $\frac1x\sum\mu$ itself. The contrast is $a=\mu$ and $a=\lambda$, with correlation $\frac1x\sum\mu^2=0.6079$ (note $\mu\lambda=\mu^2$).

What distinguishes the two groups is complexity. Each sequence of the first group is produced by a simple deterministic machine: the number of different blocks of length $L$ it contains grows only polynomially in $L$ (it has *zero topological entropy*). Sarnak's conjecture (2010) says that every such sequence is Möbius-disjoint, that is $\frac1x\sum_{n\le x}\mu(n)a(n)\to0$. It is a theorem for periodic sequences (the prime number theorem in progressions), for $e(n\alpha)$ (Davenport; Exercise 5.7 treats the minor arcs) and for all automatic sequences, Thue–Morse and Rudin–Shapiro included (Müllner, 2017); it is open in general. The Möbius function itself has positive entropy: every pattern of signs and zeros compatible with the obvious divisibility constraints is expected to appear, and the Chowla conjecture (Tao's logarithmic version is cited in the chapter) makes this precise. For a programmer the moral is concrete: no finite automaton reading the binary digits of $n$ can predict $\mu(n)$ better than chance.